# Model inference in Altair

This notebook displays the active `plot_model_inference` implementation and renders the complete participant-by-time confidence figure for the EDA + HR + Pupil model.

In [1]:
from pathlib import Path

if Path.cwd().stem == "notebooks":
    %cd ..

%load_ext autoreload
%autoreload 2

/Users/visser/drive/PhD/Code/pain-measurement


In [2]:
import inspect
import tomllib
from pathlib import Path

import altair as alt
import joblib
from IPython.display import Code, display

from src.plots_altair import plot_model_inference, style_figure
from src.data.database_manager import DatabaseManager

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Cached inference results

In [3]:
configuration_path = Path("src/experiments/measurement/measurement_config.toml")
with configuration_path.open("rb") as file:
    stimulus_seeds = tomllib.load(file)["stimulus"]["seeds"]

feature_set = "eda_raw_heart_rate_pupil"
inference_step_size_ms = 1000
seed_key = "_".join(map(str, stimulus_seeds))
cache_path = Path(".cache/model_inference") / (
    f"{feature_set}_inference_probabilities_"
    f"seeds_{seed_key}_step_size_{inference_step_size_ms}.joblib"
)
model_inference_results = joblib.load(cache_path)

database = DatabaseManager()
with database:
    ratings = database.get_table("Feature_Data")

{
    "feature_set": feature_set,
    "stimulus_seeds": stimulus_seeds,
    "sample_duration_ms": model_inference_results["sample_duration_ms"],
    "inference_step_size_ms": inference_step_size_ms,
}

{'feature_set': 'eda_raw_heart_rate_pupil',
 'stimulus_seeds': [133,
  243,
  265,
  396,
  467,
  658,
  681,
  743,
  806,
  841,
  870,
  952],
 'sample_duration_ms': 7000,
 'inference_step_size_ms': 1000}

## Rendered Altair figure

In [4]:
model_inference_chart = plot_model_inference(
    model_inference_results["probabilities"],
    sample_duration_ms=model_inference_results["sample_duration_ms"],
    classification_threshold=0.9,
    step_size_ms=inference_step_size_ms,
    display_step_size_ms=1000,
    seeds_to_plot=stimulus_seeds,
    only_decreases=True,
    only_non_decreases=False,
    ncols=2,
    width=500,
    height=110,
    stimulus_scale=0.5,
    stimulus_linewidth=1.5,
    ratings_df=ratings,
    column_spacing=20,
    row_spacing=10,
    title=None,
)
style_figure(model_inference_chart)

alt.VConcatChart(...)

Fig S12: EDA, 0.85
Fig S13: HR, 0.6
Fig S14: Pupil, 0.5
Fig S15: EDA + HR, 0.85
Fig S16: EDA + Pupil, 0.9
Fig S17: Facial Expression, 0.7
Fig S18: Combined w/o EEG: 0.9
Fig S19: EEG: 0.65
Fig S20: EEG + EDA: 0.9
Fig S21: Combined 0.9
Fig S22: EDA + HR + Pupil for Non-Decreases, 0.90
Fig S23: EDA + HR + Pupil for Decreases and Non-Decreases, 0.50